# Imports

In [1]:
from cyvcf2 import VCF
import numpy as np
import pandas as pd

# Calculate PCs

In [ ]:
# Read VCF
vcf = VCF("../data/ps3_gwas.vcf.gz")
samples = vcf.samples
n_samples = len(samples)

# Collect SNPs into list
genotype_list = []
for variant in vcf:
    g = np.array([gt[0] + gt[1] if gt[0] >= 0 and gt[1] >= 0 else np.nan for gt in variant.genotypes])
    genotype_list.append(g)

# Shape: (n_samples, n_snps)
X = np.vstack(genotype_list).T

# Remove SNPs that are entirely missing
valid = ~np.isnan(X).all(axis=0)
X = X[:, valid]

# Estimate allele frequencies from observed genotypes: (num_snps,)
p = np.nanmean(X, axis=0) / 2

# Expected variance of genotype under Hardy-Weinberg: (num_snps,)
var = 2*p*(1-p)

# Remove monomorphic SNPs (0 variance in population)
valid = (~np.isnan(var)) & (var > 0)
X = X[:, valid]
p = p[valid]
var = var[valid]

# Standardize: (n_samples, n_snps)
Z = (X-2*p) / np.sqrt(var)

# Calculate GRM numerator
Z[np.isnan(Z)] = 0 # Missing genotypes should contribute zero to numerator
GRM_numerator = Z @ Z.T # Dot product of individuals' standardized genotype vectors measuring genetic similarity (n_samples, n_samples)

# Calculate GRM denominator
observed = ~np.isnan(X) # Create mask for observed samples: (n_samples, n_snps)
obs_float = observed.astype(np.float64) # Convert to float: (n_samples, n_snps)
pair_counts = obs_float @ obs_float.T # Calculate how many SNPs two individuals both have genotype data for: (n_samples, n_samples)

# Avoid division by zero
pair_counts[pair_counts == 0] = 1

# Create GRM matrix
GRM = GRM_numerator / pair_counts

# PCA by eigendecomposition of GRM
k = 3
eigenvals_all, eigenvecs_all = np.linalg.eigh(GRM)

# eigh returns ascending order
eigenvals = eigenvals_all[-k:][::-1]
PCs = eigenvecs_all[:, -k:][:, ::-1]


In [3]:
# Save eigenvectors
pcs_df = pd.DataFrame(PCs, columns=[f"PC{i+1}" for i in range(k)])
pcs_df.insert(0, "IID", samples)
pcs_df.insert(0, "FID", samples)
pcs_df.to_csv("../python_results/eigenvec.txt", sep=" ", header=False, index=False)

# Save eigenvalues
eval_df = pd.DataFrame({"PC": [f"PC{i+1}" for i in range(k)], "eigenvalue": eigenvals})
eval_df.to_csv("../python_results/eigenval.txt", sep="\t", index=False)